# 08 - Reader: Text and Image Test Runs

This notebook runs one reader setting on the 120 test questions, with the settings of notebook 07 unchanged. It is run once per setting, and each run saves one answer file.

It runs on Google Colab. We use the same GPU type for every setting.

This run gives the model the evidence pages of each question as images, to test how well it reads a page from a picture when the right page is known.

In [1]:
SETTING = "gold_image"  # gold_text, gold_image, retrieved_text, retrieved_image or closed_book

In [2]:
!git clone https://github.com/dogamentese/financebench-rag.git
%cd financebench-rag
!pip install -q -U transformers accelerate
!pip install -q pymupdf
!python scripts/download_data.py

Cloning into 'financebench-rag'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 71 (delta 29), reused 57 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 739.90 KiB | 2.75 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/financebench-rag
[1/84] 3M_2018_10K
[2/84] 3M_2022_10K
[3/84] 3M_2023Q2_10Q
[4/84] ACTIVISIONBLIZZARD_2019_10K
[5/84] ADOBE_2015_10K
[6/84] ADOBE_2016_10K
[7/84] ADOBE_2017_10K
[8/84] ADOBE_2022_10K
[9/84] AES_2022_10K
[10/84] AMAZON_2017_10K
[11/84] AMAZON_2019_10K
[12/84] AMCOR_2020_10K
[13/84] AMCOR_2022_8K_dated-2022-07-01
[14/84] AMCOR_2023Q2_10Q
[15/84] AMCOR_2023Q4_EARNINGS
[16/84] AMCOR_2023_10K
[17/84] AMD_2015_10K
[18/84] AMD_2022_10K
[19/84] AMERICANEXPRESS_2022_10K
[20/84] AMERICANWATERWORKS_2020_10K
[21/84] AMERICANWATERWORKS_2021_10K
[22/84] AMERICANWATERWORKS_2022_10K
[23/84] BESTBUY_2017_10K
[24/84] BESTBUY_2019_10K
[25/84

In [3]:
import sys
sys.path.append("/content/financebench-rag/src")

## 1. Run the setting

We use the settings we determined before, for the prompt, token limit and image resolution.
Retrieved settings get the top 5 pages of fused + rerank from notebook 06.

In [5]:
import json
import time

import torch

from financebench_rag.data import REPO, gold_pages, load_questions, load_split
from financebench_rag.reader import Reader

questions = load_questions()
split = load_split()
test = [q for q in questions if q["financebench_id"] in set(split["test"])]
retrieved = {fid: [tuple(page) for page in ranked[:5]] for fid, ranked in json.load(open(REPO / "results" / "test_retrieval_hybrid.json")).items()}
print(f"{len(test)} test questions, setting {SETTING}, GPU {torch.cuda.get_device_name()}")

reader = Reader()


def pages_for(setting, q):
    if setting.startswith("gold"):
        return sorted(gold_pages(q))
    if setting.startswith("retrieved"):
        return retrieved[q["financebench_id"]]
    return []


mode = "image" if SETTING.endswith("image") else "text"
answers = {}
start = time.time()
for i, q in enumerate(test, start=1):
    answers[q["financebench_id"]] = reader.ask(q["question"], pages_for(SETTING, q), mode)
    print(f"{i}/{len(test)}", end="\r")
print(f"{SETTING} done in {(time.time() - start) / 60:.1f} min")

120 test questions, setting gold_image, GPU Tesla T4


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

... (4 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


gold_image done in 67.2 min


## 2. Save the answers

The file is downloaded right away, since files on Colab disappear when the session ends.

In [6]:
path = REPO / "results" / f"test_answers_{SETTING}.json"
path.write_text(json.dumps(answers, indent=1) + "\n")
from google.colab import files
files.download(str(path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 3. Quick check

The same checks as notebook 07, for this setting. A refusal counts only when the final answer is exactly `not found`.

In [7]:
import pandas as pd

from financebench_rag.grading import is_refusal, numeric_match

rs = list(answers.values())
metrics = [q for q in test if q["question_type"] == "metrics-generated"]
pd.Series({
    "no final answer line": sum(r["final"] is None for r in rs),
    "hit token limit": sum(r["tokens"] == reader.max_new_tokens for r in rs),
    "not found": sum(is_refusal(r["final"]) for r in rs),
    "mean seconds": round(sum(r["seconds"] for r in rs) / len(rs), 1),
    f"metrics correct of {len(metrics)}": sum(numeric_match(answers[q["financebench_id"]]["final"] or "", q["answer"]) for q in metrics),
}, name=SETTING)

,gold_image
no final answer line,11.0
hit token limit,11.0
not found,8.0
mean seconds,33.4
metrics correct of 40,35.0


35 of the 40 metrics answers are correct (high percentage as expected). 